# Chapter 1 — Introduction
### Notebook 0 · Overview, setup and the course contract

*Book reference: Keet, *Ontology Engineering* (2nd ed.), Ch. 1*

This course rebuilds Keet's textbook as a **practice-first graduate course**. Every claim the book makes in prose, you will make in code: measure it, test it, and defend it against a grader.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

## How this chapter is organised

| # | Notebook | Book section | What you produce |
|---|---|---|---|
| 0 | `00_overview_and_setup` | — | a working environment |
| 1 | `01_what_an_ontology_looks_like` | 1.1 | a spectrum classifier and its evidence |
| 2 | `02_why_ontologies_pay_off` | 1.2 | a measured integration result (recall 0 → 1) |
| 3 | `03_what_is_an_ontology` | 1.3 | a defect scanner and a definition scorecard |
| 4 | `04_exercises` | 1.5 | autograded answers to the book's exercises |
| 5 | `05_assignment` / `05_solutions` | — | problem set: a Claude triage agent for an ontology registry — graded, judge-validated, optimised, gated |


**By the end of this notebook you can:**

1. Distinguish an ontology from a vocabulary, a taxonomy and a thesaurus **by measurement**, not by assertion.
2. Demonstrate the value of an ontology by showing a query whose answer is wrong without one.
3. Argue about competing definitions of *ontology* by applying them to real artefacts and finding where they disagree.
4. Detect and explain common modelling defects automatically.
5. Build, evaluate and optimise an agent that performs this triage, and say honestly how good it is.

## Tooling

| Tool | Role |
|---|---|
| **rdflib** | RDF graphs, Turtle, SPARQL |
| **owlrl** | pure-Python OWL 2 RL reasoner — entailment with no Java |
| **LangChain / LangGraph** | agent loops and decomposed pipelines |
| **DSPy (incl. GEPA)** | programmatic prompt optimisation |
| **Apache Jena Fuseki** | *optional* triplestore (`infra/fuseki`) |

### Running against Claude
The chapter notebooks run locally with no key. Each chapter's **problem set** calls the live Anthropic API (`claude-opus-5`): put `ANTHROPIC_API_KEY` in your shell or in `oe-course/.env` (copy `.env.example`). Every model call is billed, and every problem set states its estimated budget and asks you to report a cost next to every score. Fuseki is optional: without it the course uses an in-memory rdflib store.

In [ ]:
import sys, os, json
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / 'oe_course').is_dir():
        sys.path.insert(0, str(candidate)); break
import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

In [ ]:
sys.path.insert(0, str(Path.cwd()))          # so ch01_toolkit imports
import ch01_toolkit as ch1
from oe_course import ontology as ont
from oe_course.data import corpus
import pandas as pd
pd.set_option("display.width", 120)

### Self-test: does the labelled corpus still agree with the detectors?

The course ships eleven small ontologies with **hand-written** gold labels. `verify_corpus()` checks the detectors against those labels. It must print nothing — if it complains, either a detector or a label is wrong, and the exercises downstream are measuring the wrong thing.

In [ ]:
problems = corpus.verify_corpus()
print(problems or 'corpus OK: detectors agree with all hand-written labels')
print(f'{len(corpus.CORPUS)} artefacts:', ', '.join(sorted(corpus.BY_NAME)))

### Sanity check: load the African Wildlife Ontology and measure it

In [ ]:
awo = ont.load_graph(corpus.get('awo').turtle)
m = ont.graph_metrics(awo)
print(f"{m['triples']} triples, {m['classes']} classes, "
      f"{m['logical_axioms']} logical axioms, richness {m['axiom_richness']}")
assert m['classes'] > 10 and m['restrictions'] > 0